---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

## Parameters

In [1]:
# IMPORTANT PARAMETERS:
year_to_load <- "2018"
split_chunks <- 5
rows_to_show <- 10

# Input:
to_read <- FALSE
to_split <- TRUE
to_sample <- TRUE

# Output:
to_write <- TRUE
to_group <- TRUE

# Debug:
to_profvis <- FALSE
to_view_checks <- TRUE
to_view_checks_parallelized <- FALSE
to_parallelize <- TRUE

# Sample size divisor:
sample_size_divisor <- 5

drop_cols <- c(
  paste0("ICDCODE", 13:14),
  "ICCODED15",
  paste0("ICDCODE", 16:170)
)

icd_cols <- paste0("clin_icd", 1:12)
rvs_cols <- paste0("clin_rvs", 1:20)

seed <- 123
set.seed(seed)

options(future.globals.maxSize = 1024 * 1024^2)

global_seed <- seed # for parallelized operations

## Load Required Libraries & Initial Functions

In [2]:
options(verbose = FALSE)
options(warn = -1)
library(here)


here() starts at C:/Users/resur/Documents/drg-pipeline



In [3]:
# List of scripts to source in order
scripts_to_source <- c(
  "00_libraries.R",
  "01_data-formats.R",
  "02_file-paths.R"
)

# Generate and execute source commands
for (script in scripts_to_source) {
  source(here("data-cleaning", "r_scripts", script))
}

# Perform other tasks here...
tic("Total execution time:")

# Use the function to count total rows
if (file.exists(full_claims_file())) {
  total_rows <- fread(full_claims_file(), select = 1L, header = TRUE)[, .N]
} else {
  total_rows <- fread(full_claims_file(), select = 1L, header = TRUE)[, .N]
}
print(paste("Total Rows via fread:", total_rows))

sample_size_divisor <- 25

if (to_split) {
  sample_size <- ceiling(total_rows / split_chunks / sample_size_divisor)
} else {
  sample_size <- ceiling(total_rows / sample_size_divisor)
}

# List of scripts to source in order
scripts_to_source <- c(
  "03_general-functions.R",
  "04_main-functions.R",
  "05_io-functions.R",
  "06_icd-functions.R",
  "07_rvs-functions.R",
  "08_pdx-functions.R",
  "09_grouper-functions.R",
  "10_timing-functions.R",
  "11_debug-functions.R"
)

# Generate and execute source commands
for (script in scripts_to_source) {
  source(here("data-cleaning", "r_scripts", script))
}


[1] "Total Rows via fread: 11777674"


In [4]:
options(warn = 1)


## Load Mapping Data

In [5]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read, Process, Export Data (Looping through all parts)

In [6]:
all_parts_summaries <- list()
all_parts_statistics <- list()

if (to_profvis) {
  p <- profvis({
    split_and_save_chunks()
    for (part in 1:split_chunks) {
      dt <- read_and_process_chunk(part)
      if (!is.null(dt) && nrow(dt) > 0) {
        dt <- parallelize_and_summarize_data(dt)
        write_intermediate_file(dt)
        group_data(dt)
      } else {
        print(paste("No data to process for part:", part))
      }
    }
    combine_and_print_summaries()
  })
  htmlwidgets::saveWidget(
    p,
    file = here("git-ignored-files", "profvis", "profvis.html"),
    selfcontained = TRUE
  )
} else {
  split_and_save_chunks()
  for (part in 1:split_chunks) {
    dt <- read_and_process_chunk(part)
    if (!is.null(dt) && nrow(dt) > 0) {
      dt <- parallelize_and_summarize_data(dt)
      write_intermediate_file(dt)
      group_data(dt)
    } else {
      print(paste("No data to process for part:", part))
    }
  }
  combine_and_print_summaries()
}

[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."
[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."
[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."
[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."
[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."
Rename Success:
 TRUE 



Table: ICD Replacements 1

|old_code |new_code | count|
|:--------|:--------|-----:|
|J18.92   |J1892    | 29806|
|A09.9    |A099     | 15842|
|N39.0    |N390     | 13150|
|A97.1    |A971     |  6747|
|K29.1    |K291     |  5379|
|J45.90   |J4590    |  5169|
|I10.1    |I101     |  4868|
|I10.9    |I109     |  4601|
|P36.9    |P369     |  3582|
|A97.0    |A970     |  3576|


Table: ICD Replacements 2

|old_code |new_code | count|
|:--------|:--------|-----:|
|I21.9    |I219     |   193|
|E86.1  

## Runtime Estimation

### Stop Timer

In [7]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 80.69 sec elapsed


### Calculate Speed

In [8]:
if (to_sample) {
  total_rows <- nrow(dt) * split_chunks * sample_size_divisor
} else {
  total_rows <- nrow(dt) * split_chunks
}
print_time_estimates(dt, total_time, total_rows)

Time spent (total) for 471.1k rows:  80.69 sec  (actual)
Time spent (t/row) for 471.1k rows:  0.17 msec (actual)
Time spent (total) for 11.8m rows: 33.62 min  (estimate)


## Debugging

In [9]:
library(here)
source(here("data-cleaning", "r_scripts", "11_debug-functions.R"))

input_path <- here("data-cleaning", "r_scripts")
output_file <- paste0(here("data-cleaning", "everything", "everything.R"))

concatenate_r_files(input_path, output_file)


To extract all code portions of this ipynb file (run in VS Code terminal):

jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning.ipynb